> **Chapter 7, Part 4** | Bridge notebook. **Focus:** what a vector store actually does once embeddings exist.


# Vector Stores and Similarity Search

By the time learners reach this point, they usually know what an embedding is but not what to do with one. That is the gap this notebook closes.

A vector store is not magical. At minimum, it is a structure that keeps vectors, document identifiers, and metadata together so that similarity search remains attached to something usable.

## Outputs

- a tiny in-memory vector index
- cosine similarity over normalized vectors
- top-k retrieval with optional metadata filtering
- the vocabulary needed for the later Chapter 10 lab

## Supporting reading

- Chapter 7.1 and 7.2 for embedding intuition and storage choices
- Chroma clients: https://docs.trychroma.com/docs/run-chroma/clients
- Qdrant quickstart: https://qdrant.tech/documentation/quick-start/

## Failure note

If you cannot tell me which document a vector belongs to, you do not have retrieval yet. You have orphaned coordinates.

## How I would debug this

Print the full ranked list for one query and inspect both the score and the metadata. Similarity without context is usually where confusion starts.


In [ ]:
import numpy as np
import pandas as pd

rows = pd.DataFrame(
    [
        {"chunk_id": "parks-1", "title": "Accessible trail guide", "domain": "parks", "vector": np.array([0.91, 0.30, 0.10])},
        {"chunk_id": "parks-2", "title": "Shuttle stop instructions", "domain": "parks", "vector": np.array([0.82, 0.48, 0.12])},
        {"chunk_id": "finance-1", "title": "Portfolio volatility memo", "domain": "finance", "vector": np.array([0.12, 0.90, 0.38])},
        {"chunk_id": "finance-2", "title": "Dividend yield snapshot", "domain": "finance", "vector": np.array([0.20, 0.86, 0.42])},
        {"chunk_id": "gov-1", "title": "Golden record policy", "domain": "governance", "vector": np.array([0.32, 0.24, 0.92])},
        {"chunk_id": "gov-2", "title": "Reference data standard", "domain": "governance", "vector": np.array([0.28, 0.18, 0.96])},
    ]
)

rows["vector"] = rows["vector"].map(lambda value: value / np.linalg.norm(value))
rows[["chunk_id", "title", "domain"]]


In [ ]:
def cosine_similarity(left, right):
    return float(np.dot(left, right) / (np.linalg.norm(left) * np.linalg.norm(right)))


class InMemoryVectorIndex:
    def __init__(self, frame):
        self.frame = frame.copy()

    def search(self, query_vector, top_k=3, filters=None):
        query_vector = query_vector / np.linalg.norm(query_vector)
        candidates = self.frame
        filters = filters or {}

        for key, value in filters.items():
            candidates = candidates[candidates[key] == value]

        ranked = candidates.assign(
            score=candidates["vector"].map(lambda vector: cosine_similarity(query_vector, vector))
        ).sort_values("score", ascending=False)

        return ranked[["chunk_id", "title", "domain", "score"]].head(top_k)


index = InMemoryVectorIndex(rows)
query = np.array([0.85, 0.44, 0.14])
index.search(query, top_k=3)


In [ ]:
query = np.array([0.30, 0.20, 0.93])
print('semantic only')
display(index.search(query, top_k=3))

print('metadata filtered to governance')
display(index.search(query, top_k=3, filters={"domain": "governance"}))


## What this notebook should clarify

A vector store is not merely a database column with floats in it. It is the combination of:

- vectors
- stable IDs
- metadata
- a search interface that returns ranked items rather than anonymous points

That is why Chapter 10 can treat Chroma as an implementation detail without losing the concept. The concept is the contract between vectors, metadata, and retrieval.

## Exercise

1. add a `collection` field and filter on it
2. change `query` and compare the semantic-only versus metadata-filtered ranking
3. add a second score column such as keyword overlap, then compare what a hybrid ranking would need
